In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

data = {
    'text': [
        'I love this product', 'This is amazing', 'Great service', 'Best purchase ever', 'Wonderful experience',
        'Fantastic quality', 'So happy with this', 'Excellent work', 'I really like it', 'Perfect',
        'It is great', 'Highly recommend', 'Such a joy', 'So impressed', 'Outstanding',
        'I hate this', 'This is terrible', 'Worst experience', 'Very disappointed', 'Waste of money',
        'Horrible quality', 'Do not buy', 'Awful', 'Poor service', 'I dislike this',
        'It is bad', 'Never again', 'Total waste', 'Very poor', 'Not good at all'
    ],
    'sentiment': ['POSITIVE']*15 + ['NEGATIVE']*15
}
df = pd.DataFrame(data)

X = df['text']
y = df['sentiment']

# Stratified split — keeps 50/50 class balance in test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline: vectorizer fit only on train data (no leakage)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf',   LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Cross-validation for reliability on small dataset
cv_scores = cross_val_score(pipeline, X, y, cv=5)
print(f"5-Fold CV Accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")

# Custom predictions
test_sentences = ['I love this!', 'This is terrible.']
predictions = pipeline.predict(test_sentences)

print("\n--- Custom Test Predictions ---")
for sentence, pred in zip(test_sentences, predictions):
    print(f"  '{sentence}' -> {pred}")

--- Classification Report ---
              precision    recall  f1-score   support

    NEGATIVE       0.75      1.00      0.86         3
    POSITIVE       1.00      0.67      0.80         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6

5-Fold CV Accuracy: 0.60 (+/- 0.08)

--- Custom Test Predictions ---
  'I love this!' -> POSITIVE
  'This is terrible.' -> NEGATIVE
